
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Delta Lake로 데이터 수집

## 환경 준비 (Databricks Free Edition)

- **카탈로그** `db_catalog`, **스키마** `db_schema` 를 사용합니다.
- 스키마 `db_schema` 아래 **`myfiles`** 볼륨을 만들고, **`employees.csv`** 를 업로드합니다.
- (다른 노트북에서 안내한 것과 동일한 데이터 레이아웃을 사용합니다.)



In [0]:
%sql
USE CATALOG db_catalog;
USE SCHEMA db_schema;

## A. 환경 구성 및 탐색


#### 1. 카탈로그 및 스키마 설정
기본 카탈로그를 **db_catalog**로 설정하고 스키마를 **db_schema**로 설정합니다. 그런 다음 사용 가능한 테이블을 보고 현재 스키마에 테이블이 없는지 확인합니다.

##### 1A. SQL 명령 사용

In [0]:
%sql
-- 기본 카탈로그 및 스키마 설정
USE CATALOG db_catalog;
USE SCHEMA db_schema;

-- 스키마에서 사용 가능한 테이블 표시
SHOW TABLES;

##### 1B. PySpark 사용

In [0]:
# 기본 카탈로그 및 스키마 설정(Spark 3.4.0 이상 필요)
spark.catalog.setCurrentCatalog("db_catalog")
spark.catalog.setCurrentDatabase("db_schema")

# 스키마에서 사용 가능한 테이블 표시
spark.catalog.listTables("db_schema")

#### 2. 사용 가능한 파일 보기
스키마의 **myfiles** 볼륨에서 사용 가능한 파일을 봅니다. **employees.csv** 파일만 사용할 수 있는지 확인합니다.

**참고:** 볼륨의 데이터를 참조할 때는 항상 다음 형식의 Unity Catalog에서 제공하는 경로를 사용합니다. */Volumes/catalog_name/schema_name/volume_name/*.

In [0]:
spark.sql(f"LIST '/Volumes/db_catalog/db_schema/myfiles/' ").display()

## B. Delta Lake 수집 기술
**목적:** 다양한 방법을 사용하여 파일에서 **employees.csv**  Delta 테이블을 만듭니다.

- CREATE TABLE AS (`CTAS`)
- UPLOAD UI(`User Interface`)
- COPY INTO
- AUTOLOADER (`Overview only`, `outside the scope of this module`)

#### 1. CREATE TABLE (CTAS)
1. 이전 데모와 유사한 CREATE TABLE AS 문을 사용하여 파일에서 **employees.csv** 테이블을 만듭니다. 쿼리를 실행하고 테이블이 **current_employees_ctas** 성공적으로 만들어졌는지 확인합니다.

In [0]:
%sql

-- 데모 목적으로 존재하는 경우 테이블을 삭제합니다.
DROP TABLE IF EXISTS current_employees_ctas;

-- CTAS를 사용하여 테이블 만들기
CREATE TABLE current_employees_ctas
AS
SELECT ID, FirstName, Country, Role 
FROM read_files(
  '/Volumes/db_catalog/db_schema/myfiles/',
  format => 'csv',
  header => true,
  inferSchema => true
 );

-- 스키마에서 사용 가능한 테이블 표시
SHOW TABLES;

2. **current_employees_ctas** 테이블을 쿼리하고 4개의 행과 4개의 열이 포함되어 있는지 확인합니다.

In [0]:
%sql
SELECT *
FROM current_employees_ctas;

#### 2. 업로드 UI
데이터 추가 UI를 사용하면 다양한 소스에서 Databricks로 데이터를 수동으로 로드할 수 있습니다.

1. 볼륨에서 파일을 수동으로 다운로드 **employees.csv** 하려면 다음 단계를 완료하십시오.

   a. 왼쪽 탐색 모음에서 카탈로그 아이콘을 선택합니다. 

   b. 카탈로그 **(db_catalog)** 를 클릭하세요.

   c. 새로 고침 아이콘을 선택하여 **db_catalog** 카탈로그를 새로 고칩니다.

   d. **db_catalog** 카탈로그를 확장합니다. 카탈로그 내에 다양한 스키마(데이터베이스)가 표시되어야 합니다.

   e. 스키마를 확장합니다. 첫 번째 셀의 설정 노트에서 스키마를 찾을 수 있습니다. 스키마에는 **테이블**과 **볼륨**이 포함되어 있습니다.

   f. **볼륨**을 확장한 다음 **myfiles**를 확장합니다. **myfiles** 볼륨에는 이름이 지정된 **employees.csv** 단일 CSV 파일이 포함되어야 합니다. 

   g. **employees.csv** 파일 오른쪽에 있는 케밥 메뉴를 클릭하고 **파일 다운로드**를 선택하세요. 그러면 CSV 파일이 브라우저의 다운로드 폴더로 저장됩니다.

2. 다음 단계를 완료하여 스키마에 **employees.csv**파일을 수동으로 업로드합니다. 이것은 로컬 파일을 Databricks로 로드하는 것을 모방합니다.

   a. 탐색 모음에서 스키마를 선택합니다. 

   b. 스키마 옆에 있는 줄임표(점 3개) 아이콘을 클릭하고 **카탈로그 탐색기에서 열기**를 선택합니다.

   c. **만들기** 드롭다운 아이콘 ![create_drop_down](../Includes/images/create_drop_down.png)을 선택한 후 **테이블**을 선택합니다.

   d. 브라우저의 사용 가능한 섹션에서 이전에 다운로드한 **employees.csv**을 선택하거나, **찾아보기**를 선택하고 다운로드 폴더로 이동하여 **employees.csv** 파일을 선택합니다.

3. UPLOAD UI를 사용하여 Delta 테이블을 생성하려면 다음 단계를 완료하십시오.

   a. UI에서 테이블이 카탈로그 **db_catalog**의 스키마 **db_schema**에 생성되는지 확인합니다. 

   b. **테이블 이름** 아래에 테이블의 **employees_ui** 이름을 지정합니다.

   c. 화면 하단의 **테이블 생성** 버튼을 선택하여 테이블을 생성합니다.

   d. 테이블이 성공적으로 생성되었는지 확인합니다. 그런 다음 카탈로그 탐색기 브라우저를 닫습니다.

**예제**
<br></br>

![](https://velog.velcdn.com/images/newnew_daddy/post/cfeba2b7-a09d-4c76-91a2-be2f0ea07553/image.png)

4. SHOW TABLES 문을 사용하여 스키마에서 사용 가능한 테이블을 봅니다. **employees_ui** 테이블이 생성되었는지 확인합니다. 


In [0]:
%sql
SHOW TABLES;

5. 마지막으로 테이블을 쿼리하여 내용을 검토합니다.

**참고**: UPLOAD UI를 사용하여 테이블을 업로드하지 않은 경우 이름을 **current_employees_ui**로 지정하지 않은 경우 오류가 반환됩니다.

In [0]:
%sql
SELECT * 
FROM current_employees_ui;

####3. COPY INTO
**employees.csv** 파일을 사용하여 [COPY INTO](https://docs.databricks.com/en/sql/language-manual/delta-copy-into.html) 문을 통해 테이블을 생성합니다.

문은 `COPY INTO` 파일 위치의 데이터를 Delta 테이블로 증분 방식으로 로드합니다. 이는 재시도 가능하고 멱등성 작업입니다. 이미 로드된 원본 위치의 파일은 건너뜁니다. 이는 파일이 로드된 이후 수정된 경우에도 마찬가지입니다.

1. **current_employees_copyinto**라는 빈 테이블을 만들고 열 데이터 유형을 정의합니다.

**참고:** 열이 없는 빈 테이블을 만들고 `COPY INTO`을 사용하여 스키마를 발전시킬 수도 있습니다.

In [0]:
%sql
-- 데모 목적으로 존재하는 경우 테이블을 삭제합니다.
DROP TABLE IF EXISTS current_employees_copyinto;

-- 열 데이터 형식이 있는 빈 테이블 만들기
CREATE TABLE current_employees_copyinto (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
);

2. `COPY INTO` 명령문을 사용하여 Unity Catalog에서 제공하는 경로를 사용하여 **myfiles** 볼륨(현재는 파일만 **employees.csv** 존재함)에서 모든 파일을 로드합니다. 데이터가 **current_employees_copyinto** 테이블에 로드되었는지 확인합니다.

    다음을 확인합니다.
    - **num_affected_rows** 는 4입니다.
    - **num_inserted_rows** 는 4입니다.
    - **num_skipped_correct_files** 0입니다.

In [0]:
spark.sql(f'''
COPY INTO current_employees_copyinto
  FROM '/Volumes/db_catalog/db_schema/myfiles/'
  FILEFORMAT = CSV
  FORMAT_OPTIONS (
      'header' = 'true', 
      'inferSchema' = 'true'
    )
  ''').display()

3. 테이블을 **current_employees_copyinto** 쿼리하고 4개의 행이 모두 Delta 테이블에 올바르게 복사되었는지 확인합니다.

In [0]:
%sql
SELECT * 
FROM current_employees_copyinto;

4. 명령문을 다시 실행 `COPY INTO` 하고 그것이 이미 로드된 볼륨에서 데이터를 다시 추가하지 않았는지 확인합니다. 기억하십시오, `COPY INTO` 재시도 가능하고 멱등성 작업입니다 — 이미 로드된 소스 위치의 파일은 건너뜁니다.   
    - **num_affected_rows** 0입니다.
    - **num_inserted_rows** 0입니다.
    - **num_skipped_correct_files** 0입니다.



In [0]:
spark.sql(f'''
COPY INTO current_employees_copyinto
  FROM '/Volumes/db_catalog/db_schema/myfiles/'
  FILEFORMAT = CSV
  FORMAT_OPTIONS (
      'header' = 'true', 
      'inferSchema' = 'true'
    )
  ''').display()

5. 아래 스크립트를 실행하여 **myfiles** 볼륨에 **employees2.csv** 라는 이름의 추가 CSV 파일을 생성합니다. 결과를 확인하고 볼륨에 이제 두 개의 CSV 파일이 포함되어 있는지 확인하세요: 원래의 **employees.csv** 파일과 새로운 **employees2.csv** 파일입니다.

In [0]:
%python
# COPY INTO 증분 실습용 employees2.csv 생성 (ID/FirstName/Country/Role 형식)
path = "/Volumes/db_catalog/db_schema/myfiles/employees2.csv"
csv = """ID,Firstname,Country,Role
99,LabUserTwo,DE,Analyst
100,LabUserThree,FR,Engineer
"""
dbutils.fs.put(path, csv, overwrite=True)
print("작성 완료:", path)

6. 새 **employees2.csv** 파일을 직접 쿼리합니다. CSV 파일에 2개의 행만 있는지 확인합니다.

In [0]:
%sql
SELECT 
  ID, 
  FirstName, 
  Country, 
  Role 
FROM read_files(
  '/Volumes/db_catalog/db_schema/myfiles/employees2.csv',
  format => 'csv',
  header => true,
  inferSchema => true
 );

7. 볼륨의 경로를 사용하여 `COPY INTO` 문을 다시 실행합니다. 새 **employees2.csv** 파일의 2개 행만 **current_employees_copyinto** 테이블에 추가됩니다.

    - **num_affected_rows** 는 2입니다.
    - **num_inserted_rows** 는 2입니다.
    - **num_skipped_correct_files** 0입니다.

In [0]:
spark.sql(f'''
COPY INTO current_employees_copyinto
  FROM '/Volumes/db_catalog/db_schema/myfiles/'
  FILEFORMAT = CSV
  FORMAT_OPTIONS (
      'header' = 'true', 
      'inferSchema' = 'true'
    )
  ''').display()

8. 업데이트된 **current_employees_copyinto** 테이블을 보고 새 데이터가 추가되어 6개의 행이 포함되어 있는지 확인합니다.

In [0]:
%sql
SELECT * 
FROM current_employees_copyinto;

9. 테이블의 이력을 봅니다. 3가지 버전이 있음을 유의하십시오.
    - **버전 0** 은 CREATE TABLE 문에 의해 생성된 초기 빈 테이블입니다.
    - **버전 1** 은 파일을 Delta 테이블에 로드한 `COPY INTO` 첫 번째 **employees.csv** 문입니다.
    - **버전 2** 는 새 `COPY INTO` 파일을 Delta 테이블에만 로드한 두 번째 **employees2.csv** 문입니다.

In [0]:
%sql
DESCRIBE HISTORY current_employees_copyinto;

#### 4. Auto Loader

**참고: Auto Loader는 이 과정의 범위를 벗어납니다.**

Auto Loader 추가 설정 없이 클라우드 스토리지에 도착하는 새로운 데이터 파일을 점진적이고 효율적으로 처리합니다.

![](https://velog.velcdn.com/images/newnew_daddy/post/e1bd8085-ae79-4c22-8556-eab9d64a32de/image.png)

Auto Loader 사용의 주요 이점은 다음과 같습니다.
- 파일 상태 관리 없음: 소스는 새 파일이 클라우드 스토리지에 도착할 때 점진적으로 처리합니다. 도착한 파일에 대한 상태 정보를 관리할 필요가 없습니다.
- 확장성: 소스는 디렉토리에 있는 모든 파일을 나열할 필요 없이 클라우드 서비스와 RocksDB를 활용하여 도착하는 새 파일을 효율적으로 추적합니다. 이 접근 방식은 디렉토리에 수백만 개의 파일이 있는 경우에도 확장 가능합니다.
- 사용하기 쉬운: 소스는 파일을 증분 처리하는 데 필요한 알림 및 메시지 큐 서비스를 자동으로 설정합니다. 설정이 필요하지 않습니다.

자동 로더에 대한 자세한 내용은 문서 [Auto Loader란 무엇인가요?](https://docs.databricks.com/en/ingestion/auto-loader/index.html)를 참고하세요.

## C. 정리
1. 데모 테이블을 삭제합니다.

In [0]:
%sql
DROP TABLE IF EXISTS current_employees_ctas;
DROP TABLE IF EXISTS current_employees_ui;
DROP TABLE IF EXISTS current_employees_copyinto;
SHOW TABLES;

2. **employees2.csv** 파일을 삭제합니다.

In [0]:
## employees2.csv을 myfiles 볼륨에서 제거합니다.
dbutils.fs.rm(f"/Volumes/db_catalog/db_schema/myfiles/employees2.csv")

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>